# 03 — Gold: dim_data_source

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_data_source` |
| **Grain** | One row per DataSourceInstanceId |
| **Source** | `rpt.vwDataSourceInstance` |
| **PK** | `DataSourceInstanceId` (int) |
| **Rows** | 81 |

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_data_source"
SOURCE_TABLE = "rpt.vwDataSourceInstance"

print(f"Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()
df_src.show(10, truncate=False)

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("DataSourceInstanceId").cast("int"),
    F.col("DataSourceInstanceName").cast("string"),
    F.col("DataSourceName").cast("string"),
    F.col("SourceSystem").cast("string"),
    F.col("IsIMRSource").cast("boolean"),
    F.col("IsEnabled").cast("boolean"),
    F.col("IsDeleted").cast("boolean")
)

print(f"After column select: {df_clean.count():,} rows × {len(df_clean.columns)} cols")

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================
unknown_row = spark.createDataFrame([(
    -1, "Unknown", "Unknown", "Unknown", False, False, False
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"Added Unknown member: {df_final.count():,} rows")

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("DataSourceInstanceId").distinct().count()

print(f"DQ Checks")
print(f"   Total rows:     {total:,}")
print(f"   Duplicate PKs:  {dupes}")
assert dupes == 0
print("\nAll DQ checks passed")

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")